# Diffusion-based text generation (with the SE data set)
This notebook demonstrates diffusion-based text generation using the `instruction` and `response` columns of the `salihturkoglu/se_data_set` dataset from HuggingFace.

## 1. Download and prepare the dataset
Load the dataset with HuggingFace Datasets. For each sample, `instruction` is the input and `response` is the target text.

In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

# Load the HuggingFace dataset
dataset = load_dataset('salihturkoglu/se_data_set', split='train')

# Use all examples (877 rows)
instructions = [ex['instruction'] for ex in dataset]
responses = [ex['response'] for ex in dataset]

## 2. Build the tokenizer and vocabulary
Build a vocabulary from all text and convert the sentences into tokens.

In [ ]:
from collections import Counter

def tokenize(text):
    return text.lower().strip().split()

# Build the vocabulary
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
all_texts = instructions + responses
counter = Counter()
for text in all_texts:
    counter.update(tokenize(text))

vocab_list = [PAD_TOKEN, UNK_TOKEN] + [tok for tok, count in counter.items() if count >= 2][:10000]
vocab = {tok: idx for idx, tok in enumerate(vocab_list)}
reverse_vocab = {idx: tok for tok, idx in vocab.items()}

def encode(text):
    return [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokenize(text)]

def decode(token_ids):
    return ' '.join([reverse_vocab.get(idx, UNK_TOKEN) for idx in token_ids if idx != vocab[PAD_TOKEN]])

## 3. PyTorch Dataset and DataLoader
Convert instruction and response pairs into tensors.

In [ ]:
class InstructionResponseDataset(Dataset):
    def __init__(self, instructions, responses, vocab, max_len=64):
        self.inputs = [encode(text)[:max_len] for text in instructions]
        self.targets = [encode(text)[:max_len] for text in responses]
        self.max_len = max_len
        self.vocab = vocab

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        inp = self.inputs[idx]
        tgt = self.targets[idx]
        inp = inp + [self.vocab['<PAD>']] * (self.max_len - len(inp))
        tgt = tgt + [self.vocab['<PAD>']] * (self.max_len - len(tgt))
        return torch.tensor(inp, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

max_len = 128
dataset = InstructionResponseDataset(instructions, responses, vocab, max_len=max_len)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## 4. Diffusion process: add and remove noise

In [ ]:
def add_noise(batch, noise_level=0.5):
    noisy = batch.clone()
    mask = (torch.rand(noisy.shape) < noise_level)
    random_tokens = torch.randint(2, len(vocab), noisy.shape, device=batch.device)
    noisy[mask] = random_tokens[mask]
    return noisy

## 5. Model definition

In [ ]:
class DiffusionTextModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=2048, num_layers=8, nhead=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=nhead, dim_feedforward=hidden_dim, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, src_key_padding_mask=None):
        emb = self.embedding(x)
        out = self.transformer(emb, src_key_padding_mask=src_key_padding_mask)
        out = self.fc(out)
        return out

## 6. Training process

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DiffusionTextModel(len(vocab)).to(device)

def train_diffusion_model(model, dataloader, epochs=15, noise_level=0.5):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab['<PAD>'])
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_inputs, batch_targets in dataloader:
            batch_inputs = batch_inputs.to(device)
            batch_targets = batch_targets.to(device)
            noisy_inputs = add_noise(batch_targets, noise_level)
            mask = (batch_targets == vocab['<PAD>'])
            optimizer.zero_grad()
            outputs = model(noisy_inputs, src_key_padding_mask=mask)
            loss = criterion(outputs.view(-1, outputs.size(-1)), batch_targets.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

train_diffusion_model(model, dataloader, epochs=15, noise_level=0.5)

## 7. Text generation (producing a response with diffusion)

In [ ]:
def generate_response(model, instruction, steps=8, max_len=32):
    model.eval()
    inp = encode(instruction)[:max_len]
    inp = inp + [vocab['<PAD>']] * (max_len - len(inp))
    inp_tensor = torch.tensor([inp], dtype=torch.long, device=device)
    # Initially a completely random sequence
    generated = torch.randint(2, len(vocab), (1, max_len), device=device)
    for step in range(steps):
        mask = (generated == vocab['<PAD>'])
        with torch.no_grad():
            outputs = model(generated, src_key_padding_mask=mask)
            predicted = outputs.argmax(dim=-1)
        prob = 1.0 - (step + 1) / steps
        random_mask = (torch.rand_like(generated.float()) < prob)
        generated[random_mask] = predicted[random_mask]
    tokens = generated[0].tolist()
    return decode(tokens)

# Generate a response with a sample instruction
test_instruction = instructions[0]
print('Instruction:', test_instruction)
print('Ground-truth response:', responses[0])
print('Model Response:', generate_response(model, test_instruction))

## 9. Test: Try the Model with Any Question
In the cell below, set the `test_instruction` variable to any question to view the model's answer.

In [ ]:
# You can change the question to test here.
test_instruction = "Can I take a course from an upper grade during course registration?"

print('Instruction:', test_instruction)
print('Ground-truth response:', responses[instructions.index(test_instruction)] if test_instruction in instructions else "None")
print('Model Response:', generate_response(model, test_instruction, max_len=max_len))

## 10. Testing the Model
The cell below measures how accurately the model can produce responses on the test data. As a simple accuracy metric, it computes how closely the generated response matches the original response token by token.

In [ ]:
def evaluate_diffusion_model(model, dataset, n_samples=100, steps=8):
    model.eval()
    total = 0
    correct = 0
    for i in range(min(n_samples, len(dataset))):
        inp, tgt = dataset[i]
        inp = inp.unsqueeze(0).to(device)
        tgt = tgt.unsqueeze(0).to(device)
        generated = torch.randint(2, len(vocab), tgt.shape, device=device)
        for step in range(steps):
            mask = (generated == vocab['<PAD>'])
            with torch.no_grad():
                outputs = model(generated, src_key_padding_mask=mask)
                predicted = outputs.argmax(dim=-1)
            prob = 1.0 - (step + 1) / steps
            random_mask = (torch.rand_like(generated.float()) < prob)
            generated[random_mask] = predicted[random_mask]
        mask = (tgt != vocab['<PAD>'])
        total += mask.sum().item()
        correct += ((generated == tgt) & mask).sum().item()
    accuracy = correct / total if total > 0 else 0.0
    print(f"Test accuracy: {accuracy:.2%} ({correct}/{total})")

# Test et
evaluate_diffusion_model(model, dataset, n_samples=100, steps=8)